#### Exercise 2 - Scrape data using read_html() and build a team winning prediction model ####

- Use the given url to scrape the tables data
- Clean the data by preprocessing (remove W-L%) and converting the required columns to numeric
- Create a label column suitable for classification (1s and 0s).
- Use a classification model to predict win or lose this season

In [1]:
import sqlite3
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

#### Scrape Data directly with pd.read_html() from ESPN

In [2]:


url = "https://www.espn.com/nfl/standings"
tables = pd.read_html(url)

# Fix header assignments for the stats tables (tables 1 & 3)
stats_afc = tables[1]
stats_afc.columns = stats_afc.iloc[0]
stats_afc = stats_afc[1:].reset_index(drop=True)

stats_nfc = tables[3]
stats_nfc.columns = stats_nfc.iloc[0]
stats_nfc = stats_nfc[1:].reset_index(drop=True)

# Combine Team Name tables (0 & 2) with their corresponding Stat tables (1 & 3)
afc_df = pd.concat([tables[0], stats_afc], axis=1)
nfc_df = pd.concat([tables[2], stats_nfc], axis=1)

# Combine AFC and NFC together
df = pd.concat([afc_df, nfc_df], ignore_index=True)

#### Clean & Convert Stat Columns

In [3]:

# Extract team names cleanly using iloc[:, 0]
df["Tm"] = df.iloc[:, 0].astype(str).str.extract(r"([A-Za-z\s]+)")

# Convert stats to numeric (now 'W', 'L', 'PF', 'PA' are guaranteed column names)
for col in ["W", "L", "PF", "PA"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate Point Differential & Win Percentage
df["PD"] = df["PF"] - df["PA"]
df["W-L%"] = df["W"] / (df["W"] + df["L"])
df["SoS"] = 0.0  # Placeholder for Strength of Schedule

# Drop divider rows and invalid values
df = df.dropna(subset=["W", "L", "PF", "PA", "PD"]).copy()

#### Create Target Label 

In [4]:
# Logic is that if a team has a winning percentage greater than 0.500, it is considered a winning season (1), otherwise it's a losing season (0).
df["Winning_Season"] = (df["W-L%"] > 0.500).astype(int)

#### SQL Storage

In [5]:
conn = sqlite3.connect("NFL.db") # Connect to create a db
df.to_sql("stats", conn, if_exists="replace", index=False) # Write to db with pandas

conn.close()

#### Step 5: Model Training


In [6]:
# Extract features
features = ["PF", "PA", "PD", "SoS"]

# Split the data into features (X) and target label (y)
X = df[features]
y = df["Winning_Season"]

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (75% train, 25% test) with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = model.predict(X_test)
print(f"Model Accuracy on Test Data: {accuracy_score(y_test, y_pred):.2%}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")

Model Accuracy on Test Data: 100.00%
Confusion Matrix:
[[4 0]
 [0 4]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



#### Save scaler and model fror deployment

In [7]:
# Save the scaler for future use while deploying
joblib.dump(scaler, "scaler.pkl")

# Save the trained model to a file
joblib.dump(model, "model.pkl")



['model.pkl']